# PoliGraph model modernization

Run this notebook in Google Colab with a GPU runtime. Training data and outputs live in Google Drive, not Git. The notebook refuses to train without separate train/dev data and records versions and evaluation results with every artifact.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/poligraph-training')
NER = ROOT / 'ner'
PURPOSE = ROOT / 'purpose'
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Expected NER files:', NER / 'train.spacy', NER / 'dev.spacy')
print('Expected purpose files:', PURPOSE / 'train.jsonl', PURPOSE / 'test.jsonl')

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU before training.'
print(torch.cuda.get_device_name(0))
!git clone https://github.com/lukeblevins/PoliGraph.git /content/PoliGraph
%cd /content/PoliGraph
!python -m pip install -q -r models/colab-requirements.txt
!python -m spacy download en_core_web_md
!python -m pip freeze > {OUTPUT / 'environment.txt'}

## Train and evaluate the privacy-policy NER model

`train.spacy` and `dev.spacy` must be independent datasets. Do not use the development set for training or promotion decisions become meaningless.

In [ ]:
for path in (NER / 'train.spacy', NER / 'dev.spacy'):
    assert path.is_file(), f'Missing required dataset: {path}'
!python -m spacy init fill-config models/named-entity-recognition/base_config.cfg /content/ner-config.cfg
!python -m spacy debug data /content/ner-config.cfg --paths.train {NER / 'train.spacy'} --paths.dev {NER / 'dev.spacy'}
!python -m spacy train /content/ner-config.cfg --gpu-id 0 --output {OUTPUT / 'ner-run'} --paths.train {NER / 'train.spacy'} --paths.dev {NER / 'dev.spacy'}
!python -m spacy evaluate {OUTPUT / 'ner-run/model-best'} {NER / 'dev.spacy'} --gpu-id 0 --output {OUTPUT / 'ner-metrics.json'}

## Train and evaluate purpose classification

This uses the fork's multi-label labels and the current SetFit API. Keep the held-out test file unchanged across candidate runs.

In [ ]:
for path in (PURPOSE / 'train.jsonl', PURPOSE / 'test.jsonl'):
    assert path.is_file(), f'Missing required dataset: {path}'
!python models/purpose-classification/train_setfit.py {PURPOSE / 'train.jsonl'} {PURPOSE / 'test.jsonl'} {OUTPUT / 'purpose-model'} --metrics-output {OUTPUT / 'purpose-metrics.json'}

## Promotion gate

Review `ner-metrics.json`, `purpose-metrics.json`, and `environment.txt`. Promote only if each per-label metric and the macro score meet or exceed the recorded production baseline. After approval, package the two model directories as a versioned GitHub Release asset and update `fetch_data.py`; do not commit model binaries.